# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

# Voigt function

In [ ]:
from skewerjax.optical_depth.voigt import log10_voigt

### Numerical errors and comparison between implementations

In [ ]:
xx = jnp.linspace(-9, 9, 2**13)

sigma_low = 1e-3
sigma_high = 1.0
gamma_low = 1e-3
gamma_high = 1.0
sigma = jnp.array([sigma_low, sigma_high, sigma_low, sigma_high])
gamma = jnp.array([gamma_low, gamma_low, gamma_high, gamma_high])

Voigt_dict = {}

Voigt_dict['scipy_wofz'] = log10_voigt(xx, sigma, gamma, mode="scipy_wofz")

voigt_convolution_batched = jit(
    vmap(lambda s, g: log10_voigt(xx, s, g, mode="convolution", oversample=4), in_axes=(0, 0))
)
Voigt_dict["JAX_convolution"] = voigt_convolution_batched(sigma, gamma)

voigt_exojax_batched = jit(
    vmap(lambda s, g: log10_voigt(xx, s, g, mode="ExoJAX"), in_axes=(0, 0))
)
Voigt_dict["exojax"] = voigt_exojax_batched(sigma, gamma)

In [ ]:
P = len(sigma)
colors = np.array(plt.cm.get_cmap("tab10").colors[:P])

fig, (ax, ax_res) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw=dict(height_ratios=[2, 1]))

for ii in range(P):
    ax.plot(xx, 10**Voigt_dict['scipy_wofz'][ii], color=colors[ii], linestyle="-", linewidth=1.0)
    ax.plot(xx, 10**Voigt_dict['JAX_convolution'][ii], color=colors[ii], linestyle="--", linewidth=2.)
    ax.plot(xx, 10**Voigt_dict['exojax'][ii], color=colors[ii], linestyle=":", linewidth=3.5)

for ii in range(P):
    ax_res.plot(
        xx, 10**Voigt_dict['JAX_convolution'][ii] / 10**Voigt_dict['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle="--", linewidth=1.0
    )
    ax_res.plot(
        xx, 10**Voigt_dict['exojax'][ii] / 10**Voigt_dict['scipy_wofz'][ii] - 1,
        color=colors[ii], linestyle=":", linewidth=2.0
    )

method_handles = [
    Line2D([0], [0], color="k", lw=1,  ls="-",  label="Scipy wofz"),
    Line2D([0], [0], color="k", lw=2,  ls="--", label="JAX convolution"),
    Line2D([0], [0], color="k", lw=3.5,  ls=":",  label="exojax"),
]
leg1 = ax.legend(handles=method_handles, loc="upper right", fontsize=16)
ax.add_artist(leg1)

param_handles = []
for ii in range(P):
    lbl = rf"$\sigma={sigma[ii]:g},\ \gamma={gamma[ii]:g}$"
    param_handles.append(Line2D([0], [0], color=colors[ii], lw=6, label=lbl))

leg2 = ax.legend(handles=param_handles, loc="upper left", fontsize=16)
ax.add_artist(leg2)

ax.set_title("Voigt profiles")
ax.set_ylabel(r"$V(x;\ \sigma,\ \gamma)$")
ax.set_yscale("log")
ax.set_ylim(1e-6, 1e3)

ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res.set_xlabel("x")        # only bottom panel gets the x-label
ax_res.set_ylabel("residual")
ax_res.set_yscale("symlog", linthresh=1e-8)

major_ticks = [-1e-2, -1e-6, 0.0, 1e-6, 1e-2]
ax_res.set_yticks(major_ticks)

ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

plt.subplots_adjust(hspace=0.05)
fig.tight_layout()
plt.show()

### Autodiff

In [ ]:
xx = jnp.linspace(-9.0, 9.0, 2**12)  # frequency (or velocity) offset grid

sigma0 = 1.0
gamma0 = 0.001

In [ ]:
def voigt_scalar(x, sigma, gamma):
    return 10**log10_voigt(jnp.atleast_1d(x), sigma, gamma, mode="ExoJAX")[0]

# Derivative wrt x (ν)
dVoigt_dx = vmap(grad(voigt_scalar, argnums=0), in_axes=(0, None, None))

# Derivative wrt sigma (Gaussian width)
dVoigt_dsigma = vmap(grad(voigt_scalar, argnums=1), in_axes=(0, None, None))

# Derivative wrt sigma (Lorentzian width)
dVoigt_dgamma = vmap(grad(voigt_scalar, argnums=2), in_axes=(0, None, None))

dV_dx = dVoigt_dx(xx, sigma0, gamma0)
dV_dsigma = dVoigt_dsigma(xx, sigma0, gamma0)
dV_dgamma = dVoigt_dgamma(xx, sigma0, gamma0)

In [ ]:
delta_sigma = 0.2
sigma = jnp.linspace(sigma0 - delta_sigma, sigma0 + delta_sigma, 5)
sigma_ref_idx = len(sigma) // 2
sigma_ref = sigma[sigma_ref_idx]

V_sigma = vmap(lambda s: 10**log10_voigt(xx, s, gamma0), in_axes=(0,))(sigma)
V_sigma_ref = 10**log10_voigt(xx, sigma_ref, gamma0)

delta_gamma = 0.0009
gamma = jnp.linspace(gamma0 - delta_gamma, gamma0 + delta_gamma, 5)
gamma_ref_idx = len(gamma) // 2
gamma_ref = gamma[gamma_ref_idx]

V_gamma = vmap(lambda g: 10**log10_voigt(xx, sigma0, g), in_axes=(0,))(gamma)
V_gamma_ref = 10**log10_voigt(xx, sigma0, gamma_ref)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex='col', gridspec_kw=dict(height_ratios=[2, 1]))

ax_main_s, ax_main_g = axes[0, 0], axes[0, 1]
ax_res_s, ax_res_g = axes[1, 0], axes[1, 1]

# =============================================================================
# Sigma variation (left column)
# =============================================================================
colors_sigma = np.array(plt.colormaps["Reds"](np.linspace(0, 1, len(sigma)+2)))[1:-1]

for i in range(len(sigma)):
    ax_main_s.plot(xx, V_sigma[i], color=colors_sigma[i], linewidth=2.0, linestyle='-')

for i in range(len(sigma)):
    residual = V_sigma[i] / V_sigma_ref - 1
    ax_res_s.plot(xx, residual, color=colors_sigma[i], linewidth=2.0, linestyle='-')

# Legend
param_handles_s = []
for i, s_val in enumerate(sigma):
    param_handles_s.append(Line2D([0], [0], color=colors_sigma[i], lw=6, label=rf"$\sigma={s_val:.3g}$"))
leg2_s = ax_main_s.legend(handles=param_handles_s, loc="upper left", fontsize=16)
ax_main_s.add_artist(leg2_s)

# Styling
ax_main_s.set_title(r"Voigt profile variation with $\sigma$", pad=10)
ax_main_s.set_ylabel(r"$V(x;\ \sigma,\ \gamma_0)$")
ax_main_s.set_yscale("log")
V_min_s, V_max_s = V_sigma.min(), V_sigma.max()
ax_main_s.set_ylim(V_min_s * 0.5, V_max_s * 2.0)

ax_res_s.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res_s.set_xlabel("x")
ax_res_s.set_ylabel("residual")
ax_res_s.set_yscale("symlog", linthresh=1e-6)
major_ticks_s = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
ax_res_s.set_yticks(major_ticks_s)
ax_res_s.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res_s.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

# =============================================================================
# Gamma variation (right column)
# =============================================================================
colors_gamma = np.array(plt.colormaps["Greens"](np.linspace(0, 1, len(gamma)+2)))[1:-1]

for i in range(len(gamma)):
    ax_main_g.plot(xx, V_gamma[i], color=colors_gamma[i], linewidth=2.0, linestyle='-')

for i in range(len(gamma)):
    residual = V_gamma[i] / V_gamma_ref - 1
    ax_res_g.plot(xx, residual, color=colors_gamma[i], linewidth=2.0, linestyle='-')

# Legend
param_handles_g = []
for i, g_val in enumerate(gamma):
    param_handles_g.append(Line2D([0], [0], color=colors_gamma[i], lw=6, label=rf"$\gamma={g_val:.4g}$"))
leg2_g = ax_main_g.legend(handles=param_handles_g, loc="upper left", fontsize=16)
ax_main_g.add_artist(leg2_g)

# Styling
ax_main_g.set_title(r"Voigt profile variation with $\gamma$", pad=10)
ax_main_g.set_ylabel(r"$V(x;\ \sigma_0,\ \gamma)$")
ax_main_g.set_yscale("log")
V_min_g, V_max_g = V_gamma.min(), V_gamma.max()
ax_main_g.set_ylim(V_min_g * 0.5, V_max_g * 2.0)

ax_res_g.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res_g.set_xlabel("x")
ax_res_g.set_ylabel("residual")
ax_res_g.set_yscale("symlog", linthresh=1e-6)
major_ticks_g = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
ax_res_g.set_yticks(major_ticks_g)
ax_res_g.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
ax_res_g.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

# =============================================================================
fig.tight_layout()
fig.subplots_adjust(hspace=0.05, wspace=0.3)  # Add wspace for column separation

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

ax = axes[0]
ax.plot(xx, dV_dx)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dx")
ax.set_title("Derivative wrt x")

ax = axes[1]
ax.plot(xx, dV_dsigma)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dσ")
ax.set_title("Derivative wrt σ")

ax = axes[2]
ax.plot(xx, dV_dgamma)
ax.set_xlabel("x")
ax.set_ylabel(r"dV/dγ")
ax.set_title("Derivative wrt γ")

fig.tight_layout()
plt.show()

# Lines

In [ ]:
from skewerjax.constants.lines import get_line, LINES

### Basic Usage: Fetch a single line by ID

In [ ]:
# Get the Lyman-alpha transition
lya = get_line("HI-LyA")

print(f"ID: {lya.id}")
print(f"Species: {lya.species} (ionization state: {lya.ion}+)")
print(f"Name: {lya.name}")
print(f"Rest wavelength: {lya.lambda0_A:.4f} Å")
print(f"Oscillator strength: {lya.f_osc:.4f}")
print(f"Einstein coefficient: {lya.A21_Hz:.4e} Hz = {lya.A21_GHz:.4f} GHz")
print(f"Mass: {lya.mass_amu:.4f} amu")

### Explore the Global Line Database

In [ ]:
print(f"Total lines loaded: {len(LINES)}")

for line_id, line in sorted(LINES.items(), key=lambda x: x[1].lambda0_A):
    print(f"  {line_id:11s} λ={line.lambda0_A:8.4f} Å  f={line.f_osc:.4f} ")

# Fields

In [ ]:
%autoreload

In [ ]:
from skewerjax.fields.fields import FieldBundle, evaluate_fields_for_lines

In [ ]:
# ────────────────────────────────────────────────────────────────
# Setup: Reproduce your original arrays
# ────────────────────────────────────────────────────────────────
N_sp, N_sk, N_ppsk = 3, 7, 5  # 3 lines, 7 skewers, 5 points per skewer
N_points = N_sk * N_ppsk
species_keys = ['HI', 'SiIII']  # One per absorption line

# Build quadrivectors: [scale_factor, x, y, z]
scale_factors = jnp.broadcast_to(
    jnp.linspace(0.5, 1.0, N_sk)[:, None],  # Varies by skewer
    (N_sk, N_ppsk)
).reshape(-1)
points = jnp.column_stack([scale_factors, jnp.zeros((N_points, 3))])

# LOS directions (all along x-axis)
los_dirs = jnp.array([[1.0, 0.0, 0.0]] * N_points)

In [ ]:

# ────────────────────────────────────────────────────────────────
# Create fields
# ────────────────────────────────────────────────────────────────
bundle = FieldBundle.create_uniform(
    temp_dict={'HI': 1e4, 'SiIII': 2e4},
    coll_dict={'HI': 0.0, 'SiIII': 0.0},
    turb_dict={'HI': 15e3, 'SiIII': 15e3},
    shared_velocity=(50e3, 0.0, 0.0),  # 50 km/s bulk flow
)


In [ ]:
# ────────────────────────────────────────────────────────────────
# Evaluate fields (returns dict of arrays)
# ────────────────────────────────────────────────────────────────
fields = evaluate_fields_for_lines(bundle, species_keys, points, los_dirs)

# Reshape to (N_sp, N_sk, N_ppsk)
T_K = fields['temperature'].reshape(N_sp, N_sk, N_ppsk)
A_coll_GHz = fields['collisional'].reshape(N_sp, N_sk, N_ppsk)
b_turb_m_per_s = fields['turbulence'].reshape(N_sp, N_sk, N_ppsk)
v_los_m_per_s = fields['velocity_los'].reshape(N_sp, N_sk, N_ppsk)

# Convert to v/c
c = 2.99792458e8
v_over_c_LoS = v_los_m_per_s / c

In [ ]:
# ────────────────────────────────────────────────────────────────
# Verify
# ────────────────────────────────────────────────────────────────
print(f"Temperature shape: {T_K.shape}")  # (3, 7, 5)
print(f"Temperature range: [{T_K.min():.1f}, {T_K.max():.1f}] K")
print(f"\nTurbulence shape: {b_turb_m_per_s.shape}")
print(f"Turbulence value: {b_turb_m_per_s[0,0,0]:.1f} m/s")
print(f"\nLOS velocity shape: {v_over_c_LoS.shape}")
print(f"LOS velocity range: [{v_over_c_LoS.min():.6f}, {v_over_c_LoS.max():.6f}]")

# Line Profile

In [ ]:
from skewerjax.optical_depth.optical_depth import compute_line_profiles

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class PhysicalConstants:
    c: float = 2.99792458e8                         # speed of light      — UNITS: [m / s]
    kB_amu: float = 8314.46262102654                # Boltzman constant   — UNITS: [amu ⋅ m^2 ⋅ s^-2 ⋅ K^-1]
    qe: float = 1.60217663e-19                      # electron charge     — UNITS: [C]
    me: float = 9.1093837e-31                       # electron mass       — UNITS: [kg]
    epsilon_0: float = 8.85418782e-12               # vacuum permittivity — UNITS: [C^2 ⋅ kg^−1 ⋅ m^−3 ⋅ s^2]

# single shared instance (immutable)
phys = PhysicalConstants()

In [ ]:
load_absorption_lines = ["HI-LyA", "HI-LyB", "SiIII-1206"]
absorption_lines = [LINES[lid] for lid in load_absorption_lines]

N_sp = len(absorption_lines) # number of absroption lines ''species'' (considering different absorption lines from different species at the same time)
N_sk = 7                                                                          # number of skerwers
N_ppsk = 5                                                                        # number of points sampled per skerwer

scale_factor = jnp.ones((N_sk, N_ppsk))                                                          #       (N_sk, N_ppsk)            — UNITS: adim

v_over_c_LoS = jnp.ones((N_sp, N_sk, N_ppsk))*jnp.linspace(-0.001, 0.001, N_ppsk)[None, None, :] # (N_sp, N_sk, N_ppsk)            — UNITS: adim
T_K = jnp.ones((N_sp, N_sk, N_ppsk))*jnp.geomspace(1e0, 1e6, N_sk)[None, :, None]                # (N_sp, N_sk, N_ppsk)            — UNITS: K
A_coll_GHz = jnp.zeros((N_sp, N_sk, N_ppsk))                                                     # (N_sp, N_sk, N_ppsk)            — UNITS: GHz
b_turb_m_per_s = jnp.zeros((N_sp, N_sk, N_ppsk))                                                 # (N_sp, N_sk, N_ppsk)            — UNITS: m/s

lambdas_A = jnp.linspace(970, 1280, 2**18)                                                       #                     (N_lambda,) — UNITS: Å

number_density_inv_nm3 = jnp.ones((N_sp, N_sk, N_ppsk))                                          # (N_sp, N_sk, N_ppsk)            — UNITS: 1/nm³

In [ ]:
lambda0_A = jnp.array([line.lambda0_A for line in absorption_lines])       # (N_sp,) — UNITS: [Angs]
A21_GHz   = jnp.array([line.A21_GHz   for line in absorption_lines])       # (N_sp,) — UNITS: [GHz]
mass_amu  = jnp.array([line.mass_amu  for line in absorption_lines])       # (N_sp,) — UNITS: [amu]
f_osc     = jnp.array([line.f_osc     for line in absorption_lines])       # (N_sp,) — UNITS: [adim]

gamma_GHz = (A21_GHz[:, None, None] + A_coll_GHz) / (4.0 * jnp.pi)         # (N_sp, 1, 1) — UNITS: [GHz]

v_th_m_per_s = jnp.sqrt(2.0 * phys.kB_amu * T_K / mass_amu[:, None, None]) # (N_sp, N_sk, N_ppsk) — UNITS: [m/s]
v_eff_A_GHz = (v_th_m_per_s + b_turb_m_per_s)*10                           # (N_sp, N_sk, N_ppsk) — UNITS: [Å/ns]
sigma_GHz = v_eff_A_GHz / (2.0 * lambda0_A[:, None, None])                 # (N_sp, N_sk, N_ppsk) — UNITS: [GHz]

lambdas_A_prime = lambdas_A[None, None, None, :] * scale_factor[None, ..., None] * (1 + v_over_c_LoS[..., None])
nu_GHz = ( (1. / lambdas_A_prime) - (1. / lambda0_A[:, None, None, None]) ) * phys.c*10  # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [GHz]

phi_ns = compute_line_profiles(                            # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [ns]
    nu_GHz, sigma_GHz, gamma_GHz,
    mode="ExoJAX", compute_mode="pmap", num_gpus=None
)

# print quantities to check their numeric range (avoid potential numerical overflows)
print("nu", nu_GHz.shape, nu_GHz.min(), nu_GHz.max())
print("gamma", gamma_GHz.shape, gamma_GHz.min(), gamma_GHz.max())
print("sigma", sigma_GHz.shape, sigma_GHz.min(), sigma_GHz.max())
print("phi", phi_ns.shape, phi_ns.min(), phi_ns.max())

In [ ]:
cte_nm2_per_ns = (phys.qe**2 / (4 * phys.epsilon_0 * phys.me * phys.c))*1e3                   # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [nm² / ns]
sigma_nm2 = cte_nm2_per_ns * f_osc[:, None, None, None] * phi_ns                              # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [nm²]
integrand = sigma_nm2 * number_density_inv_nm3[..., None] * scale_factor[None, ..., None]**-2 # (N_sp, N_sk, N_ppsk, N_lambda) — UNITS: [1/nm]

print("integrand", integrand.shape, integrand.min(), integrand.max())

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(lambdas_A)           # (N_lambda,)
phi_np = np.asarray(phi_ns)                     # (N_sp, N_sk, N_ppsk, N_lambda)
T_vals = np.asarray(T_K[0, :, 0])            # (N_sk,)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_ppsk = N_ppsk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(absorption_lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ax.scatter(np.array([line.lambda0_A]), np.array([4./line.A21_GHz]), color="grey", s=50, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_sk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for temperatures (colors)
temp_handles = []
for i_sk in range(N_sk):
    lbl = rf"$T = {T_vals[i_sk]:.2g}\,\mathrm{{K}}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_sk], lw=4, linestyle="-", label=lbl))
leg_T = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_T)

# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(absorption_lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=N_sp)

fig.tight_layout()
plt.show()



# Zoom half-width around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=N_sp, figsize=(14, 6), sharex=False, sharey=True)
if N_sp == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, absorption_lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    # Plot each temperature (each skewer)
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_sk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(lambdas_A)                          # (N_lambda,)
phi_np = np.asarray(phi_ns)                                    # (N_sp, N_sk, N_ppsk, N_lambda)
v_over_c_LoS_np = np.asarray(v_over_c_LoS[0, 0])            # (N_ppsk,)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_sk = N_sk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(absorption_lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ax.scatter(np.array([line.lambda0_A]), np.array([4./line.A21_GHz]), color="grey", s=50, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_ppsk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for velocities (colors)
temp_handles = []
for i_ppsk in range(N_ppsk):
    lbl = rf"$v_\mathrm{{LOS}}/c = {v_over_c_LoS_np[i_ppsk]:.2g}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_ppsk], lw=4, linestyle="-", label=lbl))
leg_v = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_v)

# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(absorption_lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=N_sp)

fig.tight_layout()
plt.show()


# Zoom half-width around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=N_sp, figsize=(14, 6), sharex=False, sharey=True)
if N_sp == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, absorption_lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    # Plot each temperature (each skewer)
    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_ppsk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()


# tests to explore vmap, pmap and Pytrees

In [ ]:
from jax import vmap, jit, pmap
import jax.numpy as jnp
import matplotlib.pyplot as plt



def make_chi2_computer(model_fn):
    """Factory: model_fn is baked into the returned function."""
    def chi2(data, params):
        model = model_fn(data["x"], params)
        return jnp.sum((data["y"] - model)**2 / data["noise"]**2)
    return chi2

def sinusoid_model(x, params):
    return params["amp"] * jnp.sin(x + params["phase"])


chi2_sinusoid = make_chi2_computer(sinusoid_model)



# Each sightline is a dict of arrays shape (n_points,)
datasets = {
    "x": jnp.array([
        [0.0, 0.5, 1.0, 1.5, 2.0],  # Sightline 0
        [0.0, 0.7, 1.4, 2.1, 2.8],  # Sightline 1
        [0.0, 1.0, 2.0, 3.0, 4.0],  # Sightline 2
    ]),
    "y": jnp.array([
        [0.1, 0.9, 0.2, -0.7, -0.3],  # Observed flux for sightline 0
        [0.0, 0.5, 0.8, 0.3, -0.5],   # Sightline 1
        [0.1, 1.2, 0.9, 0.4, 0.1],    # Sightline 2
    ]),
    "noise": jnp.full((3, 5), 0.05)  # Constant noise for all
}

params_sine = {"amp": 1.0, "phase": 0.0}




vcompute_sinusoid = vmap(chi2_sinusoid, in_axes=(0, None))
chi2_vmap = vcompute_sinusoid(datasets, params_sine)
print(chi2_vmap)

In [ ]:
import jax

In [ ]:
# Simulate 8 GPUs (use real number if available)
n_devices = min(8, jax.local_device_count())
print(f"Available devices: {jax.local_device_count()}, Using: {n_devices}")

# Reshape data: (n_devices, n_sightlines_per_device, ...)
# Pad to multiple of n_devices if needed
n_total = datasets["x"].shape[0]
if n_total % n_devices != 0:
    pad_size = n_devices - (n_total % n_devices)
    datasets = {k: jnp.pad(v, ((0, pad_size), (0, 0))) for k, v in datasets.items()}

datasets_sharded = {
    k: jnp.reshape(v, (n_devices, -1, v.shape[-1])) 
    for k, v in datasets.items()
}
print(f"Sharded shape: {datasets_sharded['x'].shape}")  # (8, 1, 5) if 8 devices

# Create pmapped pipeline
# pmap the vmapped function: one per device
pcompute_sinusoid = pmap(
    vcompute_sinusoid,  # Re-use the vmapped version
    in_axes=(0, None),  # Shard the dataset axis 0, broadcast params
    axis_name="devices"
)

# Run on all devices
chi2_pmap = pcompute_sinusoid(datasets_sharded, params_sine)
print(f"PMAP χ² (sharded): {chi2_pmap}")  # shape: (8, 1)
print(f"PMAP χ² (flattened): {chi2_pmap.flatten()[:n_total]}")